# Phase 3.6 — BBA Training on Colab GPU

Trains `TangentScoreModel(256×4)` on 63k BBA frames using online Riemannian DSM loss.
Online training (no precomputation) is feasible on GPU because `s_exp` + `s_log` at n=28
are O(n²d) dense ops that run fast on GPU.

**Before running**: Make sure you have selected a GPU runtime.
Runtime → Change runtime type → T4 GPU (free) or A100 (Colab Pro)

**Data**: Upload `bba_train.npy`, `bba_test.npy`, `bba_ref.npy` from
`riemannian-scoremd/data/processed/` to your Google Drive, or use the
direct upload cell below.

## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NO GPU — change runtime type!')

import jax
print('JAX devices:', jax.devices())

## 1. Clone repo and install dependencies

In [ ]:
# --- Edit this: your GitHub repo URL ---
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # <-- fill in
REPO_BRANCH = "main"  # or your branch name

In [ ]:
import os

# Clone
if not os.path.exists('/content/repo'):
    !git clone --branch {REPO_BRANCH} {REPO_URL} /content/repo
else:
    !cd /content/repo && git pull

# Verify structure
!ls /content/repo/riemannian-scoremd/src/

In [ ]:
# Install dependencies
# JAX with CUDA is pre-installed on Colab GPU runtimes — DO NOT reinstall jax/jaxlib
# Only install the Python packages we need
!pip install -q flax==0.10.4 optax mdtraj

In [ ]:
# Add src to path
import sys
sys.path.insert(0, '/content/repo/riemannian-scoremd/src')

# Quick import test
from manifold.pointcloud_jax import ShapeManifold
from diffusion.manifold_sde import ManifoldVP
from models.tangent_mlp import TangentScoreModel
from training.score_loss import prepare_batch
from training.train_manifold import train
print('All imports OK')

## 2. Load data

**Option A** — Upload from local machine (simple, works for any Colab).  
**Option B** — Mount Google Drive (faster if you store data there).  

Run whichever option applies.

In [ ]:
# === OPTION A: Upload from local machine ===
# Run this cell, then use the file picker to upload:
#   bba_train.npy, bba_test.npy, bba_ref.npy
# from riemannian-scoremd/data/processed/ on your laptop.

from google.colab import files
import os

os.makedirs('/content/data', exist_ok=True)
uploaded = files.upload()  # pick the 3 files
for fname, data in uploaded.items():
    with open(f'/content/data/{fname}', 'wb') as f:
        f.write(data)
    print(f'Saved {fname} ({len(data)/1e6:.1f} MB)')

In [ ]:
# === OPTION B: Load from Google Drive ===
# Skip if you used Option A.
# Put the 3 .npy files in a folder on your Drive, e.g. MyDrive/bba_data/

# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_DATA_PATH = '/content/drive/MyDrive/bba_data'  # <-- edit
# import shutil, os
# os.makedirs('/content/data', exist_ok=True)
# for f in ['bba_train.npy', 'bba_test.npy', 'bba_ref.npy']:
#     shutil.copy(f'{DRIVE_DATA_PATH}/{f}', f'/content/data/{f}')
#     print(f'Copied {f}')

In [ ]:
import numpy as np

train_data = np.load('/content/data/bba_train.npy')   # (63000, 28, 3)
test_data  = np.load('/content/data/bba_test.npy')    # (7000, 28, 3)
x_ref      = np.load('/content/data/bba_ref.npy')     # (28, 3)

N, n, d = train_data.shape
print(f'Train: {train_data.shape}  Test: {test_data.shape}')
print(f'n={n} Cα, d={d}, nd={n*d}')
print(f'Data range: [{train_data.min():.2f}, {train_data.max():.2f}] Å')

## 3. Build manifold and SDE

In [ ]:
from manifold.pointcloud_jax import ShapeManifold
from diffusion.manifold_sde import ManifoldVP

manifold = ShapeManifold(dim=d, numpoints=n, alpha=1.0, base=x_ref)
sde = ManifoldVP(manifold)

# Sanity check: s_exp round-trip
import jax
import jax.numpy as jnp

x0_test = jnp.array(train_data[0:1, None])  # (1, 1, n, d)
rng = jax.random.PRNGKey(0)
x_t, _, sigma_t = sde.marginal_prob(x0_test, 0.5, rng)
dist = float(manifold.s_distance(x0_test, x_t)[0, 0, 0])
print(f's_exp round-trip: w^delta(x_t, x_0) = {dist:.4f}  (expected ~{float(sde.alpha(0.5)*sde.sigma(0.5)):.4f})')
print('Manifold + SDE OK')

## 4. Benchmark Phase A on GPU

Time `prepare_batch` on GPU to confirm it's fast enough for online training.

In [ ]:
import time
from training.score_loss import prepare_batch

B = 64
rng = jax.random.PRNGKey(1)
x0_b = jnp.array(train_data[:B])
t_b  = jax.random.uniform(rng, (B,), minval=0.01, maxval=0.99)

# Warm up (JIT compile)
print('Warming up s_exp JIT...')
t0 = time.time()
prepare_batch(manifold, sde, x0_b, t_b, rng)
print(f'  JIT warm-up: {time.time()-t0:.2f}s')

# Benchmark
times = []
for i in range(10):
    rng, t_key, n_key = jax.random.split(rng, 3)
    x0_b = jnp.array(train_data[i*B:(i+1)*B])
    t_b  = jax.random.uniform(t_key, (B,), minval=0.01, maxval=0.99)
    t0 = time.time()
    prepare_batch(manifold, sde, x0_b, t_b, n_key)
    times.append(time.time()-t0)

mean_ms = np.mean(times)*1000
per_sample_ms = mean_ms / B
steps_per_epoch = N // B
secs_per_epoch = np.mean(times) * steps_per_epoch

print(f'Phase A (B={B}):      {mean_ms:.1f} ms/step  ({per_sample_ms:.2f} ms/sample)')
print(f'Steps/epoch:          {steps_per_epoch}')
print(f'Est. time/epoch:      {secs_per_epoch:.1f}s')
print(f'Est. 1000 epochs:     {1000*secs_per_epoch/60:.1f} min')

## 5. Train — online, full BBA dataset

In [ ]:
# --- Hyperparameters ---
# Adjust n_epochs based on the benchmark above.
# Target: loss should plateau (usually within 500–1000 epochs).
N_EPOCHS    = 1000
BATCH_SIZE  = 64
LR          = 3e-4
HIDDEN_DIMS = (256, 256, 256, 256)  # 241k params
CKPT_DIR    = '/content/runs/bba_phase36'
LOG_EVERY   = 50

In [ ]:
from models.tangent_mlp import TangentScoreModel
from training.train_manifold import train
import os

os.makedirs(CKPT_DIR, exist_ok=True)

model = TangentScoreModel(hidden_dims=HIDDEN_DIMS)

print(f'Starting training: {N_EPOCHS} epochs, B={BATCH_SIZE}, lr={LR}')
print(f'Checkpoints → {CKPT_DIR}\n')

state, history = train(
    model=model,
    manifold=manifold,
    sde=sde,
    train_data=train_data,
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LR,
    grad_clip=1.0,
    ema_decay=0.995,
    likelihood_weighting=True,
    seed=42,
    log_every=LOG_EVERY,
    ckpt_dir=CKPT_DIR,
)

## 6. Plot loss curve

In [ ]:
import matplotlib.pyplot as plt

epochs = [ep for ep, _ in history]
losses = [loss for _, loss in history]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs, losses, 'b-', linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('DSM Loss')
ax.set_title(f'BBA Riemannian DSM — {N_EPOCHS} epochs, n=28 Cα')
ax.grid(True, alpha=0.3)

# Mark first/last
finite = [(e, l) for e, l in zip(epochs, losses) if l == l]
if finite:
    first_loss = finite[0][1]
    last_loss  = finite[-1][1]
    ax.axhline(last_loss, color='r', linestyle='--', alpha=0.5,
               label=f'Final: {last_loss:.2f} ({last_loss/first_loss*100:.0f}% of initial)')
    ax.legend()

plt.tight_layout()
plt.savefig('/content/loss_curve.png', dpi=150)
plt.show()
print(f'Initial loss: {first_loss:.4f}  →  Final: {last_loss:.4f}')

## 7. Save results to Google Drive

In [ ]:
# Mount Drive and save checkpoint + loss history
# (skip if Drive already mounted from cell 2 Option B)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_SAVE_PATH = '/content/drive/MyDrive/bba_phase36'  # <-- edit if needed
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

import shutil, json

# Copy final checkpoint
shutil.copy(f'{CKPT_DIR}/ckpt_final.npz', f'{DRIVE_SAVE_PATH}/ckpt_final.npz')

# Save loss history
with open(f'{DRIVE_SAVE_PATH}/loss_history.json', 'w') as f:
    json.dump(history, f)

# Save loss curve plot
shutil.copy('/content/loss_curve.png', f'{DRIVE_SAVE_PATH}/loss_curve.png')

print(f'Saved to {DRIVE_SAVE_PATH}:')
print(f'  ckpt_final.npz  ({os.path.getsize(DRIVE_SAVE_PATH+"/ckpt_final.npz")/1e6:.1f} MB)')
print(f'  loss_history.json')
print(f'  loss_curve.png')

## 8. Quick score sanity check

Verify that the trained score points in the right direction on held-out test frames.

In [ ]:
from training.score_loss import prepare_batch

params = state['ema_params']
score_fn = lambda x_flat, t_col: model.apply(params, x_flat, t_col)

# Sample 32 test frames at t=0.5
B_test = 32
rng = jax.random.PRNGKey(99)
rng, t_key, n_key = jax.random.split(rng, 3)
x0_test = jnp.array(test_data[:B_test])
t_test  = jax.random.uniform(t_key, (B_test,), minval=0.2, maxval=0.8)

x_t, s_true = prepare_batch(manifold, sde, x0_test, t_test, n_key)

# Score predictions
s_pred_flat = score_fn(x_t.reshape(B_test, n*d), t_test.reshape(B_test, 1))
s_pred = s_pred_flat.reshape(B_test, n, d)

# Cosine similarity between s_pred and s_true (after flattening)
def cosine_sim(a, b):
    a_flat = a.reshape(len(a), -1)
    b_flat = b.reshape(len(b), -1)
    dots   = (a_flat * b_flat).sum(axis=-1)
    norms  = jnp.linalg.norm(a_flat, axis=-1) * jnp.linalg.norm(b_flat, axis=-1)
    return dots / (norms + 1e-8)

cos_sims = cosine_sim(s_pred, s_true)
print(f'Score cosine similarity (t in [0.2, 0.8]):')
print(f'  mean = {float(cos_sims.mean()):.4f}')
print(f'  std  = {float(cos_sims.std()):.4f}')
print(f'  min  = {float(cos_sims.min()):.4f}')
print()
print('(>0.5 = model pointing in the right direction; ~1.0 = near-perfect recovery)')